In [1]:
#!pip install faiss-gpu
#!pip install faiss-cpu

In [2]:
!where python
!where pip

C:\Users\DELL\anaconda3\envs\Langchain\python.exe
C:\Users\DELL\anaconda3\python.exe
C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\python.exe
C:\Users\DELL\anaconda3\envs\Langchain\Scripts\pip.exe
C:\Users\DELL\anaconda3\Scripts\pip.exe


In [3]:
import sys
print(sys.executable)
#!{sys.executable} -m pip install faiss-cpu
#!{sys.executable} -m pip install langchain==0.3.27 langgraph==0.5.4 langsmith==0.4.4 langchain-community langchain-google-genai python-dotenv

C:\Users\DELL\anaconda3\envs\Langchain\python.exe


In [4]:
import faiss
print('FAISS working')

FAISS working


In [5]:
#1.Importing libraries:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
#from langchain.embeddings import OpenAIEmbeddings
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import StructuredTool
from pydantic.v1 import BaseModel

#2.Loading API KEYS:
load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

#3.LLM:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

#4.Tool:
qa_prompt=PromptTemplate.from_template('Answer user queries clearly: {query}')
qa_chain=LLMChain(llm=llm, prompt=qa_prompt)

qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.invoke,
    description='A basic LLM that answers users queries clearly')

#5.Defining memory:
embedding=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2-preview')
retriever=FAISS.from_texts(['initial memory'],embedding).as_retriever()
memory=VectorStoreRetrieverMemory(retriever=retriever,memory_key='chat_history',return_messages=True)

C:\Users\DELL\anaconda3\envs\Langchain\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\DELL\anaconda3\envs\Langchain\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.ai.generativelanguage_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.ai.generativelanguage_v1beta past that date.
  warnings.warn(message, FutureWarning)
C:\Users\DELL\AppData\Local\Temp\ipykernel_14136\1201887

In [6]:
#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)

C:\Users\DELL\AppData\Local\Temp\ipykernel_14136\1674440501.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agent(



First question: 


> Entering new AgentExecutor chain...
Action: Simple QA
Action Input: {"query": "What is Langchain?"}
Observation: {'query': '{"query": "What is Langchain?"}', 'text': "LangChain is an open-source framework designed to simplify the development of applications powered by large language models (LLMs). It provides a structured way to combine LLMs with other sources of computation or data, enabling developers to build more complex and powerful applications than a standalone LLM could achieve.\n\n**Key Concepts and Components of LangChain:**\n\n1.  **Models:** Abstractions for various LLM providers (e.g., OpenAI, Hugging Face, Anthropic) and different types of models (chat models, text models, embedding models).\n2.  **Prompts:** Tools for managing, optimizing, and composing prompts for LLMs, including prompt templates and output parsers.\n3.  **Chains:** Sequences of calls to LLMs or other utilities. Chains allow you to combine multiple components (e.g., an LLM call fol

In [7]:
#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)


First question: 


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is Langchain?
Observation: {'query': 'What is Langchain?', 'text': 'LangChain is an **open-source framework** designed to simplify the development of applications powered by Large Language Models (LLMs).\n\nIn essence, it\'s a **toolkit or an orchestration layer** that helps developers go beyond simply sending a single prompt to an LLM. It allows you to build complex, multi-step applications by "chaining" together various components, including LLMs themselves, external data sources, and other tools.\n\nHere\'s a breakdown of what LangChain is and why it\'s so useful:\n\n### What Problem Does LangChain Solve?\n\nWhile LLMs are incredibly powerful, they have limitations:\n1.  **Lack of Real-time Data:** They are trained on historical data and don\'t inherently know about current events or specific private information.\n2.  **No Memory:** By default, an LLM 

In [8]:
search_tool=Tool(
    name='Intermediate Answer',
    func=TavilySearchResults(max_results=1).invoke,
    description='Use this tool to search for information'
)

#6.Agent wrap:
agent=initialize_agent(
    tools=[search_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)


First question: 


C:\Users\DELL\AppData\Local\Temp\ipykernel_14136\2021313840.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  func=TavilySearchResults(max_results=1).invoke,




> Entering new AgentExecutor chain...
No.
So the final answer is: Langchain is a framework designed to simplify the creation of applications using large language models (LLMs). It provides tools, components, and interfaces to build LLM-powered applications, allowing developers to chain together different components like LLMs, prompt templates, and agents to create more complex and powerful applications.

> Finished chain.

Agent response:  {'input': 'What is Langchain?', 'chat_history': 'input: What is Langchain?\noutput: LangChain is an **open-source framework** designed to simplify the development of applications powered by Large Language Models (LLMs).\n\nThink of it as a **toolkit or an orchestration layer** that helps developers build complex, multi-step applications by connecting LLMs with other components like external data sources and various tools.\n\nHere\'s why it\'s useful:\n*   **Solves LLM limitations:** It addresses issues like LLMs not having real-time data, memory, o

In [9]:
#Tool:
qa_tool_structured=Tool(
    name="SimpleQA",
    func=lambda q: qa_chain.invoke({"query": q})["text"],
    description="Use this tool to answer general questions clearly."
)

#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool_structured],
    llm=llm,
    memory=memory,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1['output'])

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2['output'])

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3['output'])


First question: 


> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "SimpleQA",
  "action_input": "What is Langchain?"
}
```
Observation: LangChain is an **open-source framework** designed to simplify the development of applications powered by Large Language Models (LLMs).

In essence, it's a **toolkit or an orchestration layer** that helps developers go beyond simply sending a single prompt to an LLM. It allows you to build complex, multi-step applications by "chaining" together various components, including LLMs themselves, external data sources, and other tools.

Here's a breakdown of what LangChain is and why it's so useful:

### What Problem Does LangChain Solve?

While LLMs are incredibly powerful, they have limitations:
1.  **Lack of Real-time Data:** They are trained on historical data and don't inherently know about current events or specific private information.
2.  **No Memory:** By default, an LLM treats each prompt as a new interaction, forgetting previ